In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.linear_model import LogisticRegression

In [ ]:
# Generate a sample classification dataset
X, y = make_classification(n_samples=1000, n_features=4, n_informative=2, random_state=42)

In [23]:
heart_attack = pd.read_csv('./Heart Attack.csv')
heart_attack.head()

,age,gender,impluse,pressurehight,pressurelow,glucose,kcm,troponin,class
0,64,1,66,160,83,160.0,1.80,0.012,negative
1,21,1,94,98,46,296.0,6.75,1.060,positive
2,55,1,64,160,77,270.0,1.99,0.003,negative
3,64,1,70,120,55,270.0,13.87,0.122,positive
4,55,1,64,112,65,300.0,1.08,0.003,negative


In [24]:
labels = []
for item in heart_attack['class']:
    if item == 'positive':
        labels.append(1)
    else:
        labels.append(0)

labels = pd.Series(labels)

In [25]:
df = pd.concat([heart_attack[['age', 'gender', 'impluse', 'pressurehight', 'pressurelow', 'glucose', 'kcm', 'troponin']].copy(), labels.copy()], axis=1)

In [26]:
scaler = MinMaxScaler((0,1))
df['age'] = scaler.fit_transform(df[['age']])
df['gender'] = scaler.fit_transform(df[['gender']])
df['impluse'] = scaler.fit_transform(df[['impluse']])
df['pressurehight'] = scaler.fit_transform(df[['pressurehight']])
df['pressurelow'] = scaler.fit_transform(df[['pressurelow']])
df['glucose'] = scaler.fit_transform(df[['glucose']])
df['kcm'] = scaler.fit_transform(df[['kcm']])
df['troponin'] = scaler.fit_transform(df[['troponin']])

In [27]:
x = df.iloc[:, :-1]
y = df.iloc[:, 8]

In [28]:
# Split the dataset into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2)

In [29]:
# Standardize the data (important for RBFN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [30]:
# Cluster the data using K-Means to get RBF neuron centers
n_clusters = 2
kmeans = KMeans(n_clusters=n_clusters)
kmeans.fit(X_train)
centers = kmeans.cluster_centers_

c:\Users\colit\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [31]:
# Calculate the RBF activations using Gaussian kernel
gamma = 1.0  # Adjust this hyperparameter for the width of Gaussian kernel
rbf_activations_train = rbf_kernel(X_train, centers, gamma=gamma)
rbf_activations_test = rbf_kernel(X_test, centers, gamma=gamma)

In [32]:
# Train a logistic regression classifier on RBF activations
clf = LogisticRegression()
clf.fit(rbf_activations_train, Y_train)

LogisticRegression()

In [33]:
# Make predictions on the test set
y_pred = clf.predict(rbf_activations_test)

In [34]:
# Calculate accuracy
accuracy = accuracy_score(Y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.62


In [ ]:
# Plot the dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis')
plt.title("Dataset Plot")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()